# 00_open_and_inspect_svg

First step in a workflow for translating SVG text with an LLM:
- open SVG files from `svg_source_files`
- extract and inspect text elements
- exclude items not intended for translation, such as numbers and dates
- save translation units as JSON in `json_files` for the next notebook, `01_translation_svg`

In [1]:
# Set directories

from pathlib import Path

PROJECT_ROOT = Path.cwd()
SVG_SOURCE_DIR = PROJECT_ROOT / "svg_source_files"
SVG_OUTPUT_DIR = PROJECT_ROOT / "svg_output_files"
JSON_DIR = PROJECT_ROOT / "json_files"

assert SVG_SOURCE_DIR.exists(), f"Missing folder: {SVG_SOURCE_DIR}"
assert SVG_OUTPUT_DIR.exists(), f"Missing folder: {SVG_OUTPUT_DIR}"
assert JSON_DIR.exists(), f"Missing folder: {JSON_DIR}"

print("PROJECT_ROOT:", PROJECT_ROOT.name)
print("SVG_SOURCE_DIR:", SVG_SOURCE_DIR.relative_to(PROJECT_ROOT.parent))
print("JSON_DIR:", JSON_DIR.relative_to(PROJECT_ROOT.parent))
print("SVG_OUTPUT_DIR:", SVG_OUTPUT_DIR.relative_to(PROJECT_ROOT.parent))

PROJECT_ROOT: bst
SVG_SOURCE_DIR: bst\svg_source_files
JSON_DIR: bst\json_files
SVG_OUTPUT_DIR: bst\svg_output_files


In [2]:
# Display svg files in source directory

svg_paths = sorted(SVG_SOURCE_DIR.glob("*.svg"))

print("Found SVG files:", len(svg_paths))
for p in svg_paths:
    print(" -", p.name)


Found SVG files: 2
 - BibleStructureTimelineNT_2024.svg
 - BibleStructureTimelineOT_2024.svg


## Read and parse SVG files
**NOTE:** This workflow will process *all* svg files contained in the svg_source_files directory 

In [3]:
from lxml import etree
import pandas as pd
import json
import re

SVG_NS = "http://www.w3.org/2000/svg"
NS = {"svg": SVG_NS}

def localname(tag: str) -> str:
    return tag.split("}", 1)[1] if tag and tag.startswith("{") else (tag or "")

def norm_ws(s: str) -> str:
    if s is None:
        return ""
    return re.sub(r"\s+", " ", s).strip()

def parse_style(style: str):
    out = {}
    if not style:
        return out
    for part in style.split(";"):
        part = part.strip()
        if not part or ":" not in part:
            continue
        k, v = part.split(":", 1)
        out[k.strip()] = v.strip()
    return out

def group_label(g) -> str:
    return g.get("data-name") or g.get("id") or "g"

def compute_group_stack(el) -> list[str]:
    stack = []
    cur = el.getparent()
    while cur is not None:
        if localname(cur.tag) == "g":
            stack.append(group_label(cur))
        cur = cur.getparent()
    return list(reversed(stack))

def element_path(el) -> str:
    parts = []
    cur = el
    while cur is not None:
        tag = localname(cur.tag)
        if tag in ("svg", "g", "text", "tspan"):
            _id = cur.get("id")
            if _id:
                parts.append(f"{tag}#{_id}")
            else:
                parent = cur.getparent()
                if parent is None:
                    parts.append(tag)
                else:
                    same = [
                        c for c in parent
                        if hasattr(c, "tag") and localname(c.tag) == tag
                    ]
                    idx = same.index(cur) + 1  # 1-based index
                    parts.append(f"{tag}[{idx}]")
        cur = cur.getparent()
    return "/".join(reversed(parts))

def get_text_and_tspans(text_el):
    tspans = []
    chunks = []
    if text_el.text and norm_ws(text_el.text):
        chunks.append(text_el.text)

    for t in text_el.findall(".//svg:tspan", namespaces=NS):
        t_text = t.text or ""
        if norm_ws(t_text):
            chunks.append(t_text)
        tspans.append({
            "tspan_id": t.get("id"),
            "tspan_text": t_text,
            "tspan_text_norm": norm_ws(t_text),
            "x": t.get("x"),
            "y": t.get("y"),
            "dx": t.get("dx"),
            "dy": t.get("dy"),
            "style": t.get("style"),
            "class": t.get("class"),
        })
        if t.tail and norm_ws(t.tail):
            chunks.append(t.tail)

    return "".join(chunks), tspans

def extract_text_inventory(svg_path: Path) -> pd.DataFrame:
    parser = etree.XMLParser(remove_blank_text=False, recover=True, huge_tree=True)
    tree = etree.parse(str(svg_path), parser)
    root = tree.getroot()

    rows = []
    for text_el in root.xpath("//svg:text", namespaces=NS):
        full_text, tspans = get_text_and_tspans(text_el)
        full_text_norm = norm_ws(full_text)
        if not full_text_norm:
            continue

        gstack = compute_group_stack(text_el)
        style = parse_style(text_el.get("style"))

        rows.append({
            "source_file": svg_path.name,
            "text_id": text_el.get("id"),
            "group_stack": " / ".join(gstack),
            "group_depth": len(gstack),
            "text_raw": full_text,
            "text_norm": full_text_norm,
            "has_tspans": len(tspans) > 0,
            "tspan_count": len(tspans),
            "x": text_el.get("x"),
            "y": text_el.get("y"),
            "transform": text_el.get("transform"),
            "class": text_el.get("class"),
            "style": text_el.get("style"),
            "style_font_family": style.get("font-family"),
            "style_font_size": style.get("font-size"),
            "style_text_anchor": style.get("text-anchor"),
            "element_path": element_path(text_el),
            "tspans": tspans,
        })

    df = pd.DataFrame(rows).sort_values(["group_stack", "element_path"]).reset_index(drop=True)
    return df

all_dfs = []
for p in svg_paths:
    df = extract_text_inventory(p)
    all_dfs.append(df)

    out_json = JSON_DIR / f"{p.stem}.text_extract.json"
    records = df.to_dict(orient="records")
    out_json.write_text(json.dumps(records, ensure_ascii=False, indent=2), encoding="utf-8")

    print(f"{p.name}: extracted {len(df)} text elements -> {out_json.name}")

df_all = pd.concat(all_dfs, ignore_index=True) if all_dfs else pd.DataFrame()
print("Total extracted across files:", len(df_all))
df_all.head(10)


BibleStructureTimelineNT_2024.svg: extracted 67 text elements -> BibleStructureTimelineNT_2024.text_extract.json
BibleStructureTimelineOT_2024.svg: extracted 105 text elements -> BibleStructureTimelineOT_2024.text_extract.json
Total extracted across files: 172


,source_file,text_id,group_stack,group_depth,text_raw,text_norm,has_tspans,tspan_count,x,y,transform,class,style,style_font_family,style_font_size,style_text_anchor,element_path,tspans
0,BibleStructureTimelineNT_2024.svg,None,Headline,1,The Resurrection of Jesus is the Best Attested...,The Resurrection of Jesus is the Best Attested...,True,1,None,None,translate(45 586),st6,None,None,None,None,svg/g#Headline/text[1],"[{'tspan_id': None, 'tspan_text': 'The Resurre..."
1,BibleStructureTimelineNT_2024.svg,None,NT_History,1,History,History,True,1,None,None,translate(41.1 68.7),st1,None,None,None,None,svg/g#NT_History/text[1],"[{'tspan_id': None, 'tspan_text': 'History', '..."
2,BibleStructureTimelineNT_2024.svg,None,NT_History,1,John the Apostle,John the Apostle,True,1,None,None,translate(163.66 293),st10,None,None,None,None,svg/g#NT_History/text[2],"[{'tspan_id': None, 'tspan_text': 'John the Ap..."
3,BibleStructureTimelineNT_2024.svg,None,NT_History,1,Matthew the Apostle,Matthew the Apostle,True,1,None,None,translate(146.55 131),st10,None,None,None,None,svg/g#NT_History/text[3],"[{'tspan_id': None, 'tspan_text': 'Matthew the..."
4,BibleStructureTimelineNT_2024.svg,None,NT_History,1,John Mark,John Mark,True,1,None,None,translate(190.77 185),st10,None,None,None,None,svg/g#NT_History/text[4],"[{'tspan_id': None, 'tspan_text': 'John Mark',..."
5,BibleStructureTimelineNT_2024.svg,None,NT_History,1,"Luke the Physician, Companion of Paul","Luke the Physician, Companion of Paul",True,1,None,None,translate(72.97 239),st10,None,None,None,None,svg/g#NT_History/text[5],"[{'tspan_id': None, 'tspan_text': 'Luke the Ph..."
6,BibleStructureTimelineNT_2024.svg,None,NT_History,1,"Luke the Physician, Companion of Paul","Luke the Physician, Companion of Paul",True,1,None,None,translate(414.97 131),st10,None,None,None,None,svg/g#NT_History/text[6],"[{'tspan_id': None, 'tspan_text': 'Luke the Ph..."
7,BibleStructureTimelineNT_2024.svg,None,NT_History / Acts_Box,2,5Acts,5Acts,True,2,None,None,translate(402.07 99.03) scale(.97 1),st5,None,None,None,None,svg/g#NT_History/g#Acts_Box/text[1],"[{'tspan_id': None, 'tspan_text': '5', 'tspan_..."
8,BibleStructureTimelineNT_2024.svg,None,NT_History / John_Box,2,4John,4John,True,2,None,None,translate(132.07 261.03) scale(.97 1),st5,None,None,None,None,svg/g#NT_History/g#John_Box/text[1],"[{'tspan_id': None, 'tspan_text': '4', 'tspan_..."
9,BibleStructureTimelineNT_2024.svg,None,NT_History / Luke_Box,2,3Luke,3Luke,True,2,None,None,translate(150.07 207.03) scale(.97 1),st5,None,None,None,None,svg/g#NT_History/g#Luke_Box/text[1],"[{'tspan_id': None, 'tspan_text': '3', 'tspan_..."


### Human-readable summary

In [4]:
def outline(df: pd.DataFrame, max_chars: int = 90):
    for i, r in df.iterrows():
        snippet = r["text_norm"]
        if len(snippet) > max_chars:
            snippet = snippet[:max_chars-1] + "…"
        print(f"[{i:04d}] {r['group_stack']} :: {snippet}")

for fname in df_all["source_file"].unique():
    print("\n" + "="*80)
    print(fname)
    print("="*80)
    outline(df_all[df_all["source_file"] == fname].reset_index(drop=True), max_chars=110)



BibleStructureTimelineNT_2024.svg
[0000] Headline :: The Resurrection of Jesus is the Best Attested Event in History
[0001] NT_History :: History
[0002] NT_History :: John the Apostle
[0003] NT_History :: Matthew the Apostle
[0004] NT_History :: John Mark
[0005] NT_History :: Luke the Physician, Companion of Paul
[0006] NT_History :: Luke the Physician, Companion of Paul
[0007] NT_History / Acts_Box :: 5Acts
[0008] NT_History / John_Box :: 4John
[0009] NT_History / Luke_Box :: 3Luke
[0010] NT_History / Mark_Box :: 2Mark
[0011] NT_History / Matthew_Box :: 1Matthew
[0012] NT_Letters :: Letters
[0013] NT_Letters :: Half-brother of Jesus
[0014] NT_Letters :: Half-brother of Jesus
[0015] NT_Letters :: John the Apostle
[0016] NT_Letters :: Peter the Apostle
[0017] NT_Letters :: Unknown, perhaps Paul
[0018] NT_Letters :: Paul the Apostle
[0019] NT_Letters :: Paul the Apostle
[0020] NT_Letters / Colossians_Box :: 12Colossians
[0021] NT_Letters / Ephesians_Box :: 10Ephesians
[0022] NT_Letters 

### Create LLM-ready table of elements

In [5]:
import hashlib
import pandas as pd

def make_unit_key(source_file: str, element_path: str, tspan_idx: int | None, text_id: str | None, tspan_id: str | None) -> str:
    # Prefer explicit ids when available; otherwise hash a stable locator.
    parts = [source_file, element_path]
    if text_id:
        parts.append(f"text_id={text_id}")
    if tspan_id:
        parts.append(f"tspan_id={tspan_id}")
    if tspan_idx is not None:
        parts.append(f"tspan_idx={tspan_idx}")
    raw = "|".join(parts)
    return hashlib.sha1(raw.encode("utf-8")).hexdigest()

def build_translation_df(df_text: pd.DataFrame) -> pd.DataFrame:
    out = []
    for _, r in df_text.iterrows():
        source_file = r["source_file"]
        element_path = r["element_path"]
        group_stack = r["group_stack"]
        text_id = r.get("text_id")

        tspans = r.get("tspans") or []
        if tspans:
            for j, t in enumerate(tspans):
                src = t.get("tspan_text_norm") or ""
                if not src:
                    continue
                unit_key = make_unit_key(source_file, element_path, j, text_id, t.get("tspan_id"))
                out.append({
                    "unit_key": unit_key,
                    "unit_type": "tspan",
                    "source_file": source_file,
                    "group_stack": group_stack,
                    "element_path": element_path,
                    "text_id": text_id,
                    "tspan_id": t.get("tspan_id"),
                    "tspan_idx": j,
                    "source_text": src,
                })
        else:
            src = r.get("text_norm") or ""
            if not src:
                continue
            unit_key = make_unit_key(source_file, element_path, None, text_id, None)
            out.append({
                "unit_key": unit_key,
                "unit_type": "text",
                "source_file": source_file,
                "group_stack": group_stack,
                "element_path": element_path,
                "text_id": text_id,
                "tspan_id": None,
                "tspan_idx": None,
                "source_text": src,
            })

    df_units = pd.DataFrame(out)
    # helpful sorting for review
    return df_units.sort_values(["source_file", "group_stack", "element_path", "unit_type", "tspan_idx"]).reset_index(drop=True)

df_units = build_translation_df(df_all)
print("Translation units:", len(df_units))
df_units.head(20)


Translation units: 275


,unit_key,unit_type,source_file,group_stack,element_path,text_id,tspan_id,tspan_idx,source_text
0,dff2455410d362f483bc04508cbcb5753bca52e3,tspan,BibleStructureTimelineNT_2024.svg,Headline,svg/g#Headline/text[1],None,None,0,The Resurrection of Jesus is the Best Attested...
1,28f9e896771634fe5b28da1b42499b9af0f75317,tspan,BibleStructureTimelineNT_2024.svg,NT_History,svg/g#NT_History/text[1],None,None,0,History
2,f68d6449d4fb305e4ee8fbc2c63e27b3c8804cbf,tspan,BibleStructureTimelineNT_2024.svg,NT_History,svg/g#NT_History/text[2],None,None,0,John the Apostle
3,a4653f3314eea5bd559a4ee5a5d4fe270c468cde,tspan,BibleStructureTimelineNT_2024.svg,NT_History,svg/g#NT_History/text[3],None,None,0,Matthew the Apostle
4,868584564de32b98e7ea2cdfbf70b0fff67c3f03,tspan,BibleStructureTimelineNT_2024.svg,NT_History,svg/g#NT_History/text[4],None,None,0,John Mark
5,bb0c7e00990a6f81b12f907151f44957605ce468,tspan,BibleStructureTimelineNT_2024.svg,NT_History,svg/g#NT_History/text[5],None,None,0,"Luke the Physician, Companion of Paul"
6,32425388bc206477680f01e421b5f721020989c8,tspan,BibleStructureTimelineNT_2024.svg,NT_History,svg/g#NT_History/text[6],None,None,0,"Luke the Physician, Companion of Paul"
7,3e0fe360e6f5a5500826d1598f8a502d0667ace2,tspan,BibleStructureTimelineNT_2024.svg,NT_History / Acts_Box,svg/g#NT_History/g#Acts_Box/text[1],None,None,0,5
8,7aef81e2f7aec81230d0a96a083f7441fb6d6ac3,tspan,BibleStructureTimelineNT_2024.svg,NT_History / Acts_Box,svg/g#NT_History/g#Acts_Box/text[1],None,None,1,Acts
9,507e8de4928eb32856afbe875f746cbac99de364,tspan,BibleStructureTimelineNT_2024.svg,NT_History / John_Box,svg/g#NT_History/g#John_Box/text[1],None,None,0,4


### Skip numbers and dates
Comment out if you wish to have the LLM translate the numbers or dates

In [6]:
import re

def is_numericish(s: str) -> bool:
    s = s.strip().lower()
    # numbers, punctuation, simple bc/ad, ca
    return bool(re.fullmatch(r"[0-9\s\.,:\-–—/()]*", s)) or bool(re.fullmatch(r"(ca\s*)?[0-9\s\.,\-–—/]+(bc|bce|ad|ce)?", s))

def is_short_label(s: str) -> bool:
    return len(s.strip()) <= 2

df_units["skip_reason"] = ""
df_units.loc[df_units["source_text"].map(is_short_label), "skip_reason"] = "too_short"
df_units.loc[df_units["source_text"].map(is_numericish), "skip_reason"] = "numericish"

df_units_keep = df_units[df_units["skip_reason"] == ""].copy()
df_units_skip = df_units[df_units["skip_reason"] != ""].copy()

print("Keep:", len(df_units_keep), "Skip:", len(df_units_skip))
df_units_skip.head(20)


Keep: 199 Skip: 76


,unit_key,unit_type,source_file,group_stack,element_path,text_id,tspan_id,tspan_idx,source_text,skip_reason
7,3e0fe360e6f5a5500826d1598f8a502d0667ace2,tspan,BibleStructureTimelineNT_2024.svg,NT_History / Acts_Box,svg/g#NT_History/g#Acts_Box/text[1],None,None,0,5,numericish
9,507e8de4928eb32856afbe875f746cbac99de364,tspan,BibleStructureTimelineNT_2024.svg,NT_History / John_Box,svg/g#NT_History/g#John_Box/text[1],None,None,0,4,numericish
11,7ac7c029cfb58bce91eb311916dcf39ac6246c10,tspan,BibleStructureTimelineNT_2024.svg,NT_History / Luke_Box,svg/g#NT_History/g#Luke_Box/text[1],None,None,0,3,numericish
13,d7b0c6a47569811cb6fe4dc7ad7d9171f712ce67,tspan,BibleStructureTimelineNT_2024.svg,NT_History / Mark_Box,svg/g#NT_History/g#Mark_Box/text[1],None,None,0,2,numericish
15,a06936c95d08b42696b30bc1ec37b485690cf718,tspan,BibleStructureTimelineNT_2024.svg,NT_History / Matthew_Box,svg/g#NT_History/g#Matthew_Box/text[1],None,None,0,1,numericish
25,c36f193d805f1acc901fea0464a9c7a88d9aab24,tspan,BibleStructureTimelineNT_2024.svg,NT_Letters / Colossians_Box,svg/g#NT_Letters/g#Colossians_Box/text[1],None,None,0,12,numericish
27,2af68062c158e3168b98e4427be30093ef892246,tspan,BibleStructureTimelineNT_2024.svg,NT_Letters / Ephesians_Box,svg/g#NT_Letters/g#Ephesians_Box/text[1],None,None,0,10,numericish
29,a60624b59dad44d17321f59b0161e6626eecd115,tspan,BibleStructureTimelineNT_2024.svg,NT_Letters / Galatians_Box,svg/g#NT_Letters/g#Galatians_Box/text[1],None,None,0,9,numericish
31,30e9acf98e167fe5dba02bef4ef3ba3f4fae7054,tspan,BibleStructureTimelineNT_2024.svg,NT_Letters / Hebrews_Box,svg/g#NT_Letters/g#Hebrews_Box/text[1],None,None,0,19,numericish
33,d9518f5218b96f696ba702e52e69372f1263c7b8,tspan,BibleStructureTimelineNT_2024.svg,NT_Letters / James_Box,svg/g#NT_Letters/g#James_Box/text[1],None,None,0,20,numericish


In [7]:
print("Duplicate unit_keys:", df_units_keep["unit_key"].duplicated().sum())

Duplicate unit_keys: 0


### Parse to JSON and save to JSON files directory
**CAUTION:** this will overwrite any previously run/saved versions

In [8]:
import json
from pathlib import Path

out_units = JSON_DIR / "translation_units.json"

if out_units.exists():
    print("Overwriting existing file:", out_units.relative_to(PROJECT_ROOT.parent))
else:
    print("Creating new file:", out_units.relative_to(PROJECT_ROOT.parent))

chars_written = out_units.write_text(
    json.dumps(df_units_keep.to_dict(orient="records"), ensure_ascii=False, indent=2),
    encoding="utf-8"
)

print("Characters written:", chars_written)

Overwriting existing file: bst\json_files\translation_units.json
Characters written: 72202
